In [1]:
# Imports
import torch
import torch.nn as nn
import torchvision

In [2]:
# Let's load the dataset
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

size = (64, 64)
transform = torchvision.transforms.Compose([torchvision.transforms.Resize(size), torchvision.transforms.ToTensor()])
train_dataset = list(torchvision.datasets.Flowers102("/tmp/flowers", "train", transform=transform, download=True))
test_dataset = list(torchvision.datasets.Flowers102("/tmp/flowers", "test", transform=transform, download=True))

train_images = torch.stack([img for img, _ in train_dataset], dim=0).to(device)

train_labels = torch.tensor([label for _, label in train_dataset]).to(device)


In [ ]:

###################major change from binary to multiclass is the 102 which is the n number of items in the dataset###################
model = torch.nn.Linear(3 * 64 * 64, 102).to(device)
###################change loss function from binary to multiclass###################
loss = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0)

for epoch in range(1000):  
    #Compute the model output
    out = model(train_images.view(-1, 3 * 64 * 64))
    #print(out.shape)

    #Compute the loss
    loss_val = loss(out, train_labels)

    #compute the gradients and update the weights
    optimizer.zero_grad()
    loss_val.backward()
    optimizer.step()

    #Print the loss every 10 epochs to save memory and time
    if epoch % 10 == 0:
        print(f"epoch={epoch}: {loss_val.item()}")


    #Result #1
    #Range of 100 epochs
    #Learning rate of 0.001:
    #starts just above 4.6 
    #gets better to 4.50 after 100 epochs 


    #Result #2
    #Range of 1000 epochs #####this is the change from 100 to 1000 epochs#####
    #Learning rate of 0.001:
    #starts just above 4.6 
    #gets better to 3.62 after 1000 epochs 

    #Result #3
    #Range of 1000 epochs 
    #Learning rate of 0.005 #####this is the change from 0.001 to 0.005 learning rate#####:
    #starts just above 4.6 
    #gets better to 1.92 after 1000 epochs 

epoch=0: 4.66154146194458
epoch=10: 4.568488121032715
epoch=20: 4.5004167556762695
epoch=30: 4.437769889831543
epoch=40: 4.377696990966797
epoch=50: 4.319693088531494
epoch=60: 4.263591289520264
epoch=70: 4.209278583526611
epoch=80: 4.156655788421631
epoch=90: 4.105630397796631
epoch=100: 4.05611515045166
epoch=110: 4.008025646209717
epoch=120: 3.961287498474121
epoch=130: 3.9158272743225098
epoch=140: 3.8715767860412598
epoch=150: 3.8284740447998047
epoch=160: 3.786458730697632
epoch=170: 3.745476722717285
epoch=180: 3.7054760456085205
epoch=190: 3.6664087772369385
epoch=200: 3.6282312870025635
epoch=210: 3.5909006595611572
epoch=220: 3.5543785095214844
epoch=230: 3.518629312515259
epoch=240: 3.4836180210113525
epoch=250: 3.4493141174316406
epoch=260: 3.4156877994537354
epoch=270: 3.382711410522461
epoch=280: 3.3503592014312744
epoch=290: 3.3186073303222656
epoch=300: 3.2874326705932617
epoch=310: 3.256814956665039
epoch=320: 3.226733684539795
epoch=330: 3.197169780731201
epoch=340: 3

In [9]:
test_images = torch.stack([img for img, _ in test_dataset], dim=0).to(device)
test_labels = torch.tensor([label for _, label in test_dataset]).to(device)

In [ ]:
#test the model on the test set
pred_test = model(test_images.view(-1, 3 * 64 * 64))

#just 2 examples printed:
#uncalibrated probabilities for each of the 102 classes for the first 2 test images
#so chance they belong to class 0, class 1, class 2, etc. for the first 2 test images
#closest to 1 is the most likely class for each image based on the model but need a nicer way to show what the model is predicting for each image 
#to do this we add argmax(dim=1)
#print(pred_test[:2])

#add argmax to get the predicted class for each image
#print(pred_test.argmax(dim=1))
#to view just 2 examples:
#print(pred_test.argmax(dim=1)[:2])

#test what the model predicted compared to the actual labels for the first test images
print((pred_test.argmax(dim=1) == test_labels).float().mean())
#result is 0.1517 which is 15% accuracy which is better than random guessing (which would be 1/102 or about 0.0098) but still not great. We could try to improve the model by adding more layers, using a different architecture, or tuning the hyperparameters.



tensor(0.1517)


In [ ]:
def accuracy(pred: torch.Tensor, label: torch.Tensor) -> float:
    return (pred.argmax(dim=-1) == label).float().mean().item()


model = torch.nn.Linear(in_features=size[0] * size[1] * 3, out_features=102)
model = model.to(device)

loss_fn = nn.CrossEntropyLoss()
optim = torch.optim.SGD(model.parameters(), lr=2e-2)
num_epochs = 500

for epoch in range(num_epochs):
    pred = model(train_images.view(train_images.shape[0], -1))
    loss_val = loss_fn(pred, train_labels)

    optim.zero_grad()
    loss_val.backward()
    optim.step()

    if epoch % 25 == 0 or epoch == num_epochs - 1:
        print(f"{epoch =:5d}  loss = {loss_val.item():.2f}  accuracy(train) = {accuracy(pred, train_labels):.3f}")

    if epoch % 100 == 0 or epoch == num_epochs - 1:
        with torch.inference_mode():
            pred = model(test_images.view(test_images.shape[0], -1))
            print(f"   Accuracy (test): {accuracy(pred, test_labels):.3f}")